# 📈 Talk to Your Tables: Finance & Stock RAG Pipeline
**LlamaIndex `NLSQLTableQueryEngine` + Groq + LLM caching + Plotly charts**

This notebook builds a retrieval-augmented generation (RAG) pipeline over tables, step by step. It runs on two CSVs from the Dataset **[47 Companies. 10 Sectors. 5 Years of Data.](https://www.kaggle.com/datasets/sometimessubodh/bcgx-updated-2021-2025)**:

| Table | File | Grain |
|---|---|---|
| `financials` | `Financials_with_Valuation.csv` | company x fiscal year: revenue, net income, assets, liabilities, margins, P/E, P/S, P/B... |
| `stock_market` | `Stock_Market_Data.csv` | company x fiscal year: close/high/low, returns, volatility, volume, shares |

### How it works❓
Please, checkout the [Process_Flowchart](https://github.com/SUBODH-sys/Query_Tables/blob/main/flowchart.png) on GitHub cause adding here on the Markdown is increasing the Kernel size to greater than 1 Mb which is then raising the error❌`"The kernel source must be less than 1 megabyte in size."` 

## Step 1 · Install dependencies

In [1]:
!pip install -q llama-index-core llama-index-llms-groq plotly sqlalchemy pandas requests

## Step 2 · Getting Groq API key


In [2]:
import os

if not os.getenv("GROQ_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
        print("Loaded GROQ_API_KEY from Kaggle Secrets")
    except Exception:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

print("Key present:", bool(os.getenv("GROQ_API_KEY")))

Loaded GROQ_API_KEY from Kaggle Secrets
Key present: True


## Step 3 · Check the CSVs/Tables


In [3]:
import glob
import pandas as pd

csvs = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("\n".join(csvs) or "No CSVs found, attach your dataset via 'Add Input'")

fin_path = next(p for p in csvs if p.endswith("Financials_with_Valuation.csv"))
stk_path = next(p for p in csvs if p.endswith("Stock_Market_Data.csv"))
fin_raw, stk_raw = pd.read_csv(fin_path), pd.read_csv(stk_path)
print("\nfinancials   :", fin_raw.shape, "| stock_market:", stk_raw.shape)
fin_raw.head(3)
print('\n')
stk_raw.head(3)

/kaggle/input/datasets/sometimessubodh/bcgx-updated-2021-2025/10K_Fillings_2021-2025 (1).csv
/kaggle/input/datasets/sometimessubodh/bcgx-updated-2021-2025/10K_Fillings_v2.csv
/kaggle/input/datasets/sometimessubodh/bcgx-updated-2021-2025/Feature_Engineered_Data (1).csv
/kaggle/input/datasets/sometimessubodh/bcgx-updated-2021-2025/Feature_Engineered_Data_v2.csv
/kaggle/input/datasets/sometimessubodh/bcgx-updated-2021-2025/Financials_with_Valuation.csv
/kaggle/input/datasets/sometimessubodh/bcgx-updated-2021-2025/Stock_Market_Data.csv

financials   : (235, 26) | stock_market: (235, 16)




,Company,Ticker,Year,Fiscal Year End,Price Date,Close Price (USD),FY High (USD),FY Low (USD),FY Avg Close (USD),FY Price Return (%),FY Total Return (%),FY Volatility (%),Avg Daily Volume (M),"Shares (M, split-adjusted)",Shares Source,Market Cap (USD M)
0,Abbvie Inc.,ABBV,2021,2021-12-31,2021-12-31,135.40,135.93,102.30,113.07,26.36,32.43,20.16,6.82,1777.0,SEC diluted weighted avg,240606
1,Abbvie Inc.,ABBV,2022,2022-12-31,2022-12-30,161.61,174.96,131.98,148.70,19.36,24.01,22.62,6.47,1778.0,SEC diluted weighted avg,287343
2,Abbvie Inc.,ABBV,2023,2023-12-31,2023-12-29,154.97,166.55,132.51,148.02,-4.11,-0.23,19.75,5.50,1773.0,SEC diluted weighted avg,274762


## Step 4 · CSV → SQLite

- Column names are cleaned for SQL: `Revenue Growth (%)` → `revenue_growth_pct`, `P/E Ratio` → `p_e_ratio`. LLMs write far more reliable SQL against clean names.

In [4]:
# Finance & Stock RAG pipeline: LlamaIndex NLSQLTableQueryEngine + Groq
import glob
import hashlib
import json
import os
import re
import sqlite3
import threading
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import plotly.express as px
from pydantic import PrivateAttr
from sqlalchemy import create_engine

from llama_index.core import PromptTemplate, Settings, SQLDatabase
from llama_index.core.embeddings import MockEmbedding
from llama_index.core.llms import ChatMessage, ChatResponse, CompletionResponse, MessageRole
from llama_index.core.prompts import PromptType
from llama_index.core.query_engine import NLSQLTableQueryEngine
from llama_index.llms.groq import Groq


ON_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path(os.getenv("WORK_DIR", "/kaggle/working" if ON_KAGGLE else "."))
DB_PATH = WORK_DIR / "finance.db"
CACHE_PATH = WORK_DIR / "llm_cache.db"

# Two models, because Groq's free-tier limits are per model (separate token budgets):
#   SQL model  -> accurate text-to-SQL           (big model)
#   TEXT model -> rewrite follow-ups, write answers, optional chart spec  (small, cheap, fast)
DEFAULT_SQL_MODEL = "openai/gpt-oss-120b"
DEFAULT_TEXT_MODEL = "openai/gpt-oss-20b"
DEFAULT_MODEL = DEFAULT_SQL_MODEL

# Fallback list; the app fetches the real list for your key with list_groq_models().
AVAILABLE_MODELS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b"]

# table name -> csv file name
CSV_FILES = {
    "financials": "Financials_with_Valuation.csv",
    "stock_market": "Stock_Market_Data.csv",
}

SAMPLE_QUESTIONS = [
    "Which 5 companies had the highest net profit margin in 2025?",
    "Plot Microsoft's total revenue and net income from 2021 to 2025",
    "Which company had the best total stock return in 2025?",
    "Show a line chart of Nvidia's closing price by year",
]


def find_csv(filename: str) -> str:
    """Look for a CSV in DATA_DIR env var, cwd, /kaggle/input/** and /kaggle/working."""
    candidates = []
    if os.getenv("DATA_DIR"):
        candidates.append(os.path.join(os.environ["DATA_DIR"], filename))
    candidates += [filename, str(WORK_DIR / filename)]
    candidates += glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"{filename} not found. Attach it as a Kaggle dataset, or set DATA_DIR / put it next to main.py."
    )


def clean_column(name: str) -> str:
    """'Revenue Growth (%)' -> 'revenue_growth_pct', 'P/E Ratio' -> 'p_e_ratio'."""
    name = name.strip().lower().replace("%", "pct")
    return re.sub(r"[^a-z0-9]+", "_", name).strip("_")


def build_database(force: bool = False) -> Path:
    """Load both CSVs into a SQLite file (one table per CSV) with SQL-friendly column names."""
    if DB_PATH.exists() and not force:
        return DB_PATH
    if DB_PATH.exists():
        DB_PATH.unlink()
    eng = create_engine(f"sqlite:///{DB_PATH}")
    for table, fname in CSV_FILES.items():
        df = pd.read_csv(find_csv(fname))
        df.columns = [clean_column(c) for c in df.columns]
        df.to_sql(table, eng, index=False, if_exists="replace")
    with eng.begin() as conn:
        conn.exec_driver_sql("CREATE INDEX IF NOT EXISTS ix_fin ON financials(ticker, year)")
        conn.exec_driver_sql("CREATE INDEX IF NOT EXISTS ix_stk ON stock_market(ticker, year)")
    eng.dispose()
    return DB_PATH

In [5]:

build_database(force=True)

import sqlite3
con = sqlite3.connect(DB_PATH)
for t in CSV_FILES:
    cols = [r[1] for r in con.execute(f"PRAGMA table_info({t})")]
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows\n  {cols}\n")
con.close()

financials: 235 rows
  ['company', 'ticker', 'sector', 'year', 'fiscal_year_end', 'total_revenue', 'net_income', 'total_assets', 'total_liabilities', 'cash_flow_from_operating_activities', 'company_code', 'revenue_growth_pct', 'net_income_growth_pct', 'total_assets_growth_pct', 'net_profit_margin_pct', 'return_on_assets_pct', 'debt_to_assets_ratio', 'implied_equity', 'debt_to_equity_ratio', 'operating_cash_flow_margin_pct', 'cash_flow_to_net_income_ratio', 'market_cap_usd_m', 'p_e_ratio', 'p_s_ratio', 'p_b_ratio', 'earnings_yield_pct']

stock_market: 235 rows
  ['company', 'ticker', 'year', 'fiscal_year_end', 'price_date', 'close_price_usd', 'fy_high_usd', 'fy_low_usd', 'fy_avg_close_usd', 'fy_price_return_pct', 'fy_total_return_pct', 'fy_volatility_pct', 'avg_daily_volume_m', 'shares_m_split_adjusted', 'shares_source', 'market_cap_usd_m']



## Step 5 · Schema context and a read-only database
- `SQLDatabase` (LlamaIndex) exposes the table schemas to the LLM.
- We add a **data dictionary** (units, NULL rules, join keys, sectors, ticker↔company map). This is the biggest accuracy win for text-to-SQL, because the model learns that revenue is in *USD millions*, that `_pct` columns are already percentages, and that tables join on `ticker + year`.
- The database is opened with `mode=ro`, so even a malicious or buggy generated query cannot change data.

In [6]:
# SCHEMA CONTEXT + READ-ONLY SQL DATABASE
def get_readonly_engine():
    """Read-only SQLite connection: the LLM can never modify the data, even if it tries."""
    return create_engine(f"sqlite:///file:{DB_PATH}?mode=ro&uri=true")


def get_sql_database() -> SQLDatabase:
    return SQLDatabase(get_readonly_engine(), include_tables=list(CSV_FILES))


def build_schema_context() -> str:
    """Business-level data dictionary that is added to the table schemas in the text-to-SQL prompt."""
    eng = get_readonly_engine()
    companies = pd.read_sql_query("SELECT DISTINCT ticker, company, sector FROM financials ORDER BY ticker", eng)
    years = pd.read_sql_query("SELECT MIN(year) AS lo, MAX(year) AS hi FROM financials", eng).iloc[0]
    sectors = sorted(companies["sector"].unique())
    pairs = "; ".join(f"{r.ticker}={r.company}" for r in companies.itertuples())
    eng.dispose()
    return (
        "DATA DICTIONARY\n"
        "- Table `financials`: one row per company per fiscal year (fundamentals + valuation multiples). "
        "Money columns (total_revenue, net_income, total_assets, total_liabilities, "
        "cash_flow_from_operating_activities, implied_equity, market_cap_usd_m) are in USD millions. "
        "Columns ending in _pct are percentages (12.5 means 12.5%). "
        "p_e_ratio is NULL when earnings are negative; p_b_ratio is NULL when equity is negative.\n"
        "- Table `stock_market`: one row per company per fiscal year (price statistics). Prices are in USD; "
        "fy_price_return_pct excludes dividends, fy_total_return_pct includes them; fy_volatility_pct is "
        "annualised volatility; avg_daily_volume_m and shares_m_split_adjusted are in millions.\n"
        "- Join the two tables on ticker AND year. Both contain market_cap_usd_m, so qualify it when joining.\n"
        f"- Years available: {int(years.lo)} to {int(years.hi)} (the 'latest year' is {int(years.hi)}).\n"
        f"- Sectors: {'; '.join(sectors)}.\n"
        f"- Companies (ticker=name): {pairs}.\n"
    )

In [7]:
print(build_schema_context())
print(get_sql_database().get_single_table_info("stock_market"))

DATA DICTIONARY
- Table `financials`: one row per company per fiscal year (fundamentals + valuation multiples). Money columns (total_revenue, net_income, total_assets, total_liabilities, cash_flow_from_operating_activities, implied_equity, market_cap_usd_m) are in USD millions. Columns ending in _pct are percentages (12.5 means 12.5%). p_e_ratio is NULL when earnings are negative; p_b_ratio is NULL when equity is negative.
- Table `stock_market`: one row per company per fiscal year (price statistics). Prices are in USD; fy_price_return_pct excludes dividends, fy_total_return_pct includes them; fy_volatility_pct is annualised volatility; avg_daily_volume_m and shares_m_split_adjusted are in millions.
- Join the two tables on ticker AND year. Both contain market_cap_usd_m, so qualify it when joining.
- Years available: 2021 to 2025 (the 'latest year' is 2025).
- Sectors: Beverages; Electronic Computers; Motor Vehicles & Passenger Car Bodies; Petroleum Refining; Pharmaceutical Preparation

## Step 6 · LLM caching
Groq calls cost latency and rate-limit quota, and text-to-SQL is deterministic at `temperature=0`, so identical prompts should never hit the API twice.

- **`LLMCache`**: a small persistent SQLite key-value store (key = SHA-256 of model + temperature + messages), with hit/miss stats, optional TTL and `clear()`. It survives notebook restarts.
- **`CachedGroq`**: subclasses LlamaIndex's `Groq` and overrides `chat()` / `complete()`. Every component that uses the LLM (SQL generation, answer synthesis, follow-up rewriting, chart-spec) is cached automatically, with no changes to the query engine.
- **Two models:** Groq's free-tier limits are *per model*, so `make_llms()` gives a big **SQL model** (`openai/gpt-oss-120b`, accurate text-to-SQL) and a small **text model** (`openai/gpt-oss-20b`, used for follow-up rewriting, written answers and the optional chart spec). Each has its own token budget, and both share one cache.
- `make_llm()` also sets `Settings.embed_model = MockEmbedding`: this engine needs no embeddings, and it stops LlamaIndex from asking for an OpenAI key.

In [8]:
# LLM CACHE (persistent, SQLite) + CachedGroq
class LLMCache:
    """Tiny persistent key-value cache for LLM responses. Survives restarts."""

    def __init__(self, path: Path = CACHE_PATH, ttl_seconds: Optional[int] = None):
        self.path = str(path)
        self.ttl = ttl_seconds
        self.hits = 0
        self.misses = 0
        self._lock = threading.Lock()
        self._exec(
            "CREATE TABLE IF NOT EXISTS llm_cache (key TEXT PRIMARY KEY, response TEXT, created REAL)"
        )

    def _exec(self, sql: str, params: tuple = ()):
        with self._lock:
            conn = sqlite3.connect(self.path, timeout=30)
            try:
                rows = conn.execute(sql, params).fetchall()
                conn.commit()
                return rows
            finally:
                conn.close()

    @staticmethod
    def make_key(model: str, temperature: float, payload: Any) -> str:
        raw = json.dumps({"m": model, "t": temperature, "p": payload}, sort_keys=True, default=str)
        return hashlib.sha256(raw.encode()).hexdigest()

    def get(self, key: str) -> Optional[str]:
        rows = self._exec("SELECT response, created FROM llm_cache WHERE key = ?", (key,))
        if rows and (self.ttl is None or time.time() - rows[0][1] < self.ttl):
            self.hits += 1
            return rows[0][0]
        self.misses += 1
        return None

    def set(self, key: str, response: str) -> None:
        self._exec(
            "INSERT OR REPLACE INTO llm_cache (key, response, created) VALUES (?, ?, ?)",
            (key, response, time.time()),
        )

    def size(self) -> int:
        return self._exec("SELECT COUNT(*) FROM llm_cache")[0][0]

    def clear(self) -> None:
        self._exec("DELETE FROM llm_cache")
        self.hits = self.misses = 0

    def stats(self) -> Dict[str, Any]:
        total = self.hits + self.misses
        return {
            "entries": self.size(),
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": round(self.hits / total, 2) if total else 0.0,
        }


THINK_REGEX = re.compile(r"<think>.*?</think>", re.S | re.I)


def clean_llm_text(text: Optional[str]) -> str:
    """Remove <think>...</think> blocks that some reasoning models (like Qwen) put in the answer."""
    return THINK_REGEX.sub("", text or "").strip()


def _require_answer(text: Optional[str]) -> str:
    out = clean_llm_text(text)
    if not out or out.lstrip().lower().startswith("<think>"):
        raise RuntimeError(
            "The model returned no final answer (a reasoning model probably used all max_tokens "
            "for thinking). Increase max_tokens in make_llm() or pick a different model."
        )
    return out


class CachedGroq(Groq):
    """Groq LLM whose chat()/complete() calls are served from LLMCache when the exact same prompt was seen."""

    _llm_cache: Optional[LLMCache] = PrivateAttr(default=None)

    def __init__(self, *args, llm_cache: Optional[LLMCache] = None, **kwargs):
        super().__init__(*args, **kwargs)
        self._llm_cache = llm_cache

    def _key(self, payload) -> str:
        return LLMCache.make_key(self.model, self.temperature, payload)

    def chat(self, messages, **kwargs) -> ChatResponse:
        cache = self._llm_cache
        if cache is None:
            return super().chat(messages, **kwargs)
        key = self._key([[m.role.value, str(m.content)] for m in messages])
        hit = cache.get(key)
        if hit is not None:
            return ChatResponse(
                message=ChatMessage(role=MessageRole.ASSISTANT, content=hit), raw={"cached": True}
            )
        resp = super().chat(messages, **kwargs)
        text = _require_answer(resp.message.content)      # never cache empty / unfinished answers
        cache.set(key, text)
        return ChatResponse(message=ChatMessage(role=MessageRole.ASSISTANT, content=text), raw=resp.raw)

    def complete(self, prompt: str, formatted: bool = False, **kwargs) -> CompletionResponse:
        cache = self._llm_cache
        if cache is None:
            return super().complete(prompt, formatted=formatted, **kwargs)
        key = self._key(["complete", prompt])
        hit = cache.get(key)
        if hit is not None:
            return CompletionResponse(text=hit, raw={"cached": True})
        resp = super().complete(prompt, formatted=formatted, **kwargs)
        text = _require_answer(resp.text)
        cache.set(key, text)
        return CompletionResponse(text=text, raw=resp.raw)


def make_llm(api_key: Optional[str] = None, model: str = DEFAULT_MODEL,
             cache: Optional[LLMCache] = None, temperature: float = 0.0,
             max_tokens: int = 2000) -> CachedGroq:
    """Create the cached Groq LLM and register it as the LlamaIndex global default."""
    api_key = api_key or os.getenv("GROQ_API_KEY")
    if not api_key:
        raise ValueError("Groq API key missing. Set GROQ_API_KEY or pass api_key=...")
    # Reasoning models spend max_tokens on hidden thinking first, so keep it generous (2000) but bounded.
    extra = {}
    if model.startswith("openai/gpt-oss"):
        extra["additional_kwargs"] = {"reasoning_effort": "low"}   # faster + far fewer tokens, fine for SQL
    llm = CachedGroq(model=model, api_key=api_key, temperature=temperature,
                     max_tokens=max_tokens, llm_cache=cache, **extra)
    Settings.llm = llm
    # NLSQLTableQueryEngine needs no embeddings; stop LlamaIndex from asking for an OpenAI key.
    Settings.embed_model = MockEmbedding(embed_dim=8)
    return llm


def make_llms(api_key: Optional[str] = None, sql_model: str = DEFAULT_SQL_MODEL,
              text_model: str = DEFAULT_TEXT_MODEL, cache: Optional[LLMCache] = None):
    """Return (sql_llm, text_llm). Both share one cache; the SQL llm becomes LlamaIndex's global default."""
    text_llm = make_llm(api_key, text_model, cache)
    sql_llm = make_llm(api_key, sql_model, cache)   # last call wins -> Settings.llm = sql_llm
    return sql_llm, text_llm

In [9]:
import time

cache = LLMCache(CACHE_PATH)
cache.clear()
sql_llm, text_llm = make_llms(cache=cache)     # uses GROQ_API_KEY
print("SQL model :", sql_llm.model, "\nText model:", text_llm.model)

for attempt in ("1st call (Groq)", "2nd call (cache)"):
    t = time.time()
    out = text_llm.complete("In one short sentence, what is a P/E ratio?").text
    print(f"{attempt}: {time.time() - t:.2f}s -> {out}")
print(cache.stats())

SQL model : openai/gpt-oss-120b 
Text model: openai/gpt-oss-20b
1st call (Groq): 2.13s -> A P/E ratio is the price of a stock divided by its earnings per share, showing how much investors pay for each dollar of earnings.
2nd call (cache): 0.00s -> A P/E ratio is the price of a stock divided by its earnings per share, showing how much investors pay for each dollar of earnings.
{'entries': 1, 'hits': 1, 'misses': 1, 'hit_rate': 0.5}


## Step 7 · NLSQLTableQueryEngine
This is the core "retriever". For each question it:
1. sends the table schemas + our data dictionary + our rules to Groq → **SQL**,
2. runs the SQL on the read-only DB,
3. sends the rows to Groq → a short natural-language **answer**.

Keeping `synthesize=False`: the **SQL model** only generates and runs SQL (1 call, schema-heavy prompt). The written answer is produced separately by `write_answer()` with the small **text model**, whose prompt contains just the question + result rows (no schema), so it is cheap. We override both default prompts: the text-to-SQL prompt adds rules (case-insensitive company match, always return identifying columns, `LIMIT`, chart-friendly output) and the answer prompt keeps replies short and unit-aware. `run_sql_df()` re-runs the generated SQL to give us a DataFrame for tables and charts.

In [10]:
# NLSQLTableQueryEngine

TEXT_TO_SQL_TMPL = (
    "Given an input question, create a syntactically correct {dialect} query to run, "
    "then look at the results and answer.\n"
    "Rules:\n"
    "- Only use the tables and columns listed below. Never invent columns.\n"
    "- Write read-only SELECT statements only.\n"
    "- Match company names case-insensitively with LOWER(company) LIKE '%name%', or use the ticker.\n"
    "- Always include identifying columns (company or ticker, and year) next to the metrics you return, "
    "so the result is self-explanatory and can be charted.\n"
    "- For 'top N' / 'highest' / 'lowest' questions use ORDER BY with LIMIT. For other plain lists use LIMIT 50.\n"
    "- Ratios and growth rates are already computed in the tables; do not recompute them unless asked.\n"
    "- If the question asks for a chart, plot or graph, do not mention charts: just return the rows "
    "needed (for example year plus the metric columns, or category plus an aggregated value).\n"
    "- Output the raw SQL after 'SQLQuery:' with no markdown fences and no explanation.\n\n"
    "Use this exact format, each item on its own line:\n"
    "Question: question here\n"
    "SQLQuery: SQL query to run\n"
    "SQLResult: result of the SQLQuery\n"
    "Answer: final answer here\n\n"
    "Only use the following tables:\n{schema}\n\n"
    "Question: {query_str}\n"
    "SQLQuery: "
)
TEXT_TO_SQL_PROMPT = PromptTemplate(TEXT_TO_SQL_TMPL, prompt_type=PromptType.TEXT_TO_SQL)

SYNTHESIS_TMPL = (
    "You are a financial data analyst. Answer the question using ONLY the SQL result below.\n"
    "Be concise (2-5 sentences). Quote numbers with units (USD millions, %, USD per share) and the year. "
    "If the result is empty, say no matching data was found. "
    "If the question asks for a chart, do not say you cannot draw one: the chart is rendered separately, "
    "so just summarise the key takeaway from the data.\n\n"
    "Question: {query_str}\n"
    "SQL: {sql_query}\n"
    "SQL result: {context_str}\n"
    "Answer: "
)
SYNTHESIS_PROMPT = PromptTemplate(SYNTHESIS_TMPL, prompt_type=PromptType.SQL_RESPONSE_SYNTHESIS_V2)


def build_query_engine(llm, sql_database: Optional[SQLDatabase] = None,
                       synthesize: bool = True) -> NLSQLTableQueryEngine:
    """synthesize=False -> SQL only (1 LLM call per question); True -> SQL + written answer (2 calls)."""
    return NLSQLTableQueryEngine(
        sql_database=sql_database or get_sql_database(),
        tables=list(CSV_FILES),
        llm=llm,
        text_to_sql_prompt=TEXT_TO_SQL_PROMPT,
        response_synthesis_prompt=SYNTHESIS_PROMPT,
        context_str_prefix=build_schema_context(),
        synthesize_response=synthesize,
    )


def write_answer(text_llm, question: str, sql: str, df: Optional[pd.DataFrame]) -> str:
    """Written answer from the SQL result, produced by the (small) text model: 1 call, no schema in the prompt."""
    if df is None or df.empty:
        return "No matching data was found."
    shown = df.head(50)
    rows = shown.to_string(index=False)[:6000]
    if len(df) > len(shown):
        rows += f"\n(showing the first {len(shown)} of {len(df)} rows)"
    prompt = SYNTHESIS_PROMPT.format(query_str=question, sql_query=sql, context_str=rows)
    return text_llm.complete(prompt).text.strip()


def clean_sql(sql: Optional[str]) -> Optional[str]:
    if not sql:
        return None
    return re.sub(r"^```(?:sql)?|```$", "", sql.strip(), flags=re.I | re.M).strip().rstrip(";")


def run_sql_df(sql: str, max_rows: int = 5000) -> pd.DataFrame:
    """Re-run the generated SQL on the read-only DB and return a DataFrame (used for tables + charts)."""
    sql = clean_sql(sql)
    if not sql or not re.match(r"^\s*(select|with)\b", sql, re.I):
        raise ValueError("Only SELECT queries are allowed.")
    eng = get_readonly_engine()
    try:
        return pd.read_sql_query(sql, eng).head(max_rows)
    finally:
        eng.dispose()

## Step 8 · Charts on demand
Charts are produced **only when the user asks** (words like *chart, plot, graph, bar, line, pie, scatter, histogram, visualize*).

How it works?:
1. The query engine returns SQL → we run it to get a DataFrame.
2. A **chart spec** (`chart_type, x, y, color, title`) is chosen. By default this is **rule-based and costs 0 LLM calls** (year + company + metric → line chart coloured by company; category + number → bar; words like *pie / bar / scatter / histogram* in the question are honoured). Optionally (`use_llm=True`) the LLM picks the spec as JSON (+1 call); it is validated against the real columns and falls back to the rules if invalid.
4. `build_figure()` turns the spec into an interactive **Plotly** figure (line, area, bar, scatter, pie, histogram, box).

Asking the LLM for a spec, not for plotting code, means there is no `exec()` of model-written code, so it's safe and robust.

In [11]:
# 5. CHARTS

CHART_REGEX = re.compile(
    r"\b(chart|plot|graph|visuali[sz]e|visuali[sz]ation|histogram|bar|pie|scatter|line|draw)\b", re.I
)
CHART_TYPES = ["line", "bar", "scatter", "pie", "area", "histogram", "box"]


def wants_chart(text: str) -> bool:
    return bool(CHART_REGEX.search(text or ""))


CHART_SPEC_INSTRUCTIONS = (
    "You are a data-visualisation assistant. Choose the best chart for the data below.\n"
    "Respond with ONLY one JSON object, no markdown, in this shape:\n"
    '{"chart_type": "line|bar|scatter|pie|area|histogram|box", "x": "<column>", '
    '"y": "<column>" or ["<column>", "<column>"] or null, "color": "<column>" or null, "title": "<short title>"}\n'
    "Rules: use exact column names. line/area for trends over year; bar for comparing categories; "
    "pie only for shares with at most 10 slices; scatter for the relationship between two numeric columns; "
    "histogram for the distribution of one numeric column (y = null). "
    "If the user named a chart type, use it. If several companies appear in one column, use it as color.\n"
)


def _title(hint: str) -> str:
    t = re.sub(r"^\s*(please\s+)?(plot|show( me)?|chart|graph|draw|visuali[sz]e|give me|create)\s+", "", hint, flags=re.I)
    return (t[:1].upper() + t[1:])[:90]


def _fallback_spec(df: pd.DataFrame, hint: str = "") -> Dict[str, Any]:
    """Rule-based chart choice (0 LLM calls). Also the fallback when an LLM spec is invalid."""
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != "year"]
    cat = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    ctype = next((t for t in CHART_TYPES if re.search(rf"\b{t}\b", hint, re.I)), None)
    title = _title(hint)
    if ctype == "pie" and cat and num:
        return {"chart_type": "pie", "x": cat[0], "y": num[0], "color": None, "title": title}
    if ctype == "histogram" and (num or "year" in df.columns):
        return {"chart_type": "histogram", "x": (num or ["year"])[0], "y": None, "color": None, "title": title}
    if "year" in df.columns and num:
        color = cat[0] if cat and df[cat[0]].nunique() > 1 else None
        return {"chart_type": ctype or "line", "x": "year", "y": num if not color else num[0],
                "color": color, "title": title}
    if cat and num:
        return {"chart_type": ctype or "bar", "x": cat[0], "y": num[0], "color": None, "title": title}
    if len(num) >= 2:
        return {"chart_type": ctype or "scatter", "x": num[0], "y": num[1], "color": None, "title": title}
    return {"chart_type": "histogram", "x": df.columns[0], "y": None, "color": None, "title": title}


def _valid_spec(spec: Any, df: pd.DataFrame) -> bool:
    if not isinstance(spec, dict) or spec.get("chart_type") not in CHART_TYPES:
        return False
    ys = spec.get("y")
    ys = [] if ys is None else (ys if isinstance(ys, list) else [ys])
    cols = [spec.get("x")] + ys + ([spec["color"]] if spec.get("color") else [])
    return all(c in df.columns for c in cols) and spec.get("x") is not None


def get_chart_spec(llm, df: pd.DataFrame, question: str, use_llm: bool = False) -> Dict[str, Any]:
    """Default: rule-based (free). use_llm=True asks the LLM to choose (+1 call), validated, with fallback."""
    if not use_llm or llm is None:
        return _fallback_spec(df, question)
    prompt = (
        f"{CHART_SPEC_INSTRUCTIONS}\nUser request: {question}\n"
        f"Columns: {list(df.columns)}\nRows: {len(df)}\nSample:\n{df.head(8).to_string(index=False)}\n\nJSON:"
    )
    try:
        text = llm.complete(prompt).text
        spec = json.loads(re.search(r"\{.*\}", text, re.S).group(0))
        if _valid_spec(spec, df):
            return spec
    except Exception:
        pass
    return _fallback_spec(df, question)


def build_figure(df: pd.DataFrame, spec: Dict[str, Any]):
    """Turn a DataFrame + chart spec into a Plotly figure."""
    df = df.copy()
    ctype, x, y, color = spec["chart_type"], spec["x"], spec.get("y"), spec.get("color")
    title = spec.get("title") or ""
    if isinstance(y, list) and len(y) == 1:
        y = y[0]
    if isinstance(y, list):
        color = None  # px cannot combine a list of y columns with a color column
    if x == "year":  # treat years as categories (no 2021.5 ticks)
        df = df.sort_values([c for c in ["year", color] if c in df.columns])
        df["year"] = df["year"].astype(str)
    common = dict(title=title, template="plotly_white")
    if ctype == "line":
        fig = px.line(df, x=x, y=y, color=color, markers=True, **common)
    elif ctype == "area":
        fig = px.area(df, x=x, y=y, color=color, **common)
    elif ctype == "bar":
        fig = px.bar(df, x=x, y=y, color=color, barmode="group", **common)
    elif ctype == "scatter":
        fig = px.scatter(df, x=x, y=y, color=color, hover_data=list(df.columns), **common)
    elif ctype == "box":
        fig = px.box(df, x=x, y=y, color=color, **common)
    elif ctype == "pie":
        fig = px.pie(df, names=x, values=y, **common)
    else:
        fig = px.histogram(df, x=x, color=color, **common)
    fig.update_layout(legend_title_text="", margin=dict(l=20, r=20, t=60, b=20))
    return fig

## Step 9 · Conversational follow-ups
`FinanceAssistant` wraps everything into one `ask()` call and keeps the chat history.

**Follow-up handling = question condensing.** Before querying, if there is history, the **text model** rewrites the new message into a *standalone* question using the last few turns. For example *"now do the same for Nvidia"* becomes *"Plot Nvidia's total revenue and net income from 2021 to 2025"*. Because SQL is generated from the standalone question, follow-ups work with the unchanged query engine.

To save LLM calls the assistant (a) only rewrites a message into a standalone question when it *looks* like a follow-up (short, or contains words like *it / that / both / same / how about*), (b) answers chart requests with just the SQL call and a rule-based chart (the chart *is* the answer), and (c) has a `llm_answers=False` fast mode (1 call per question).

`ask()` returns a `Reply` with the answer, SQL, DataFrame, optional Plotly figure, the interpreted question, and cache/latency info for the UI. Errors (rate limits, bad SQL) are returned as messages instead of crashing the app.

In [12]:
# CONVERSATIONAL ASSISTANT (follow-ups, charts, cache stats)

CONDENSE_TMPL = (
    "Given the conversation so far and a follow-up message, rewrite the follow-up as ONE standalone question "
    "about the financial / stock tables. Resolve pronouns and references such as 'it', 'that company', "
    "'the same year', 'those', 'now show'. Keep any chart request (plot, bar, line, pie...). "
    "If the follow-up is already standalone, return it unchanged. Output only the question.\n\n"
    "Conversation:\n{history}\n\n"
    "Follow-up: {question}\n"
    "Standalone question: "
)


REFERENCE_REGEX = re.compile(
    r"\b(it|its|that|this|those|these|them|they|their|same|both|also|again|too|instead|previous|above|earlier)\b"
    r"|^\s*(and|but|what about|how about)\b", re.I)


def needs_condense(question: str) -> bool:
    """Only spend an LLM call rewriting the question if it looks like a follow-up."""
    return len(question.split()) <= 6 or bool(REFERENCE_REGEX.search(question))


def auto_answer(df: Optional[pd.DataFrame], chart: bool) -> str:
    """Zero-LLM answer text used in fast mode."""
    if df is None or df.empty:
        return "No matching data was found."
    if chart:
        return f"Here is your chart ({len(df)} data points). The data table and SQL are below."
    if len(df) == 1:
        return ", ".join(f"{c}: {v}" for c, v in df.iloc[0].items())
    return f"Found {len(df)} rows. See the data table below."


@dataclass
class Reply:
    question: str
    standalone_question: str
    answer: str
    sql: Optional[str] = None
    df: Optional[pd.DataFrame] = None
    fig: Any = None
    chart_spec: Optional[Dict[str, Any]] = None
    error: Optional[str] = None
    cache_hits: int = 0
    llm_calls: int = 0
    seconds: float = 0.0


@dataclass
class FinanceAssistant:
    sql_llm: Any                      # big model: used only inside the SQL engine
    text_llm: Any                     # small model: rewrite follow-ups, written answers, optional chart spec
    sql_engine: Any                   # NLSQLTableQueryEngine(synthesize=False) -> SQL generation + execution
    llm_answers: bool = True          # False -> never write an LLM answer (fast mode, 1 call per question)
    llm_chart_spec: bool = False      # True -> text model picks chart type/axes (+1 call), else rules
    max_turns: int = 4
    history: List[Dict[str, str]] = field(default_factory=list)

    def reset(self) -> None:
        self.history.clear()

    def _history_text(self) -> str:
        lines = []
        for turn in self.history[-self.max_turns:]:
            lines.append(f"User: {turn['question']}")
            lines.append(f"Assistant: {turn['answer'][:300]}")
        return "\n".join(lines)

    def condense(self, question: str) -> str:
        if not self.history or not needs_condense(question):
            return question
        prompt = CONDENSE_TMPL.format(history=self._history_text(), question=question)
        out = self.text_llm.complete(prompt).text.strip().strip('"')
        return out.splitlines()[0] if out else question

    def ask(self, question: str) -> Reply:
        cache = getattr(self.sql_llm, "_llm_cache", None)
        h0, m0, t0 = (cache.hits, cache.misses, time.time()) if cache else (0, 0, time.time())
        standalone = question
        try:
            standalone = self.condense(question)                       # text model (only if follow-up)
            chart = wants_chart(question) or wants_chart(standalone)
            resp = self.sql_engine.query(standalone)                   # SQL model (1 call)
            sql = clean_sql((resp.metadata or {}).get("sql_query"))
            reply = Reply(question, standalone, "", sql=sql)
            if sql:
                try:
                    reply.df = run_sql_df(sql)
                except Exception as e:
                    reply.error = f"The generated SQL failed to run: {e}"
            if chart and reply.df is not None and not reply.df.empty:
                reply.chart_spec = get_chart_spec(self.text_llm, reply.df, standalone, use_llm=self.llm_chart_spec)
                reply.fig = build_figure(reply.df, reply.chart_spec)
            if reply.error:
                reply.answer = f"Sorry, I could not answer that. {reply.error}"
            elif chart or not self.llm_answers:
                reply.answer = auto_answer(reply.df, chart)            # the chart/table is the answer: 0 calls
            else:
                reply.answer = write_answer(self.text_llm, standalone, sql, reply.df)   # text model (1 call)
        except Exception as e:  # rate limits, bad key, malformed SQL, ...
            reply = Reply(question, standalone, f"Sorry, something went wrong: {e}", error=str(e))
        if cache:
            reply.cache_hits, reply.llm_calls = cache.hits - h0, cache.misses - m0
        reply.seconds = round(time.time() - t0, 2)
        self.history.append({"question": question, "standalone": standalone, "answer": reply.answer})
        return reply

In [13]:
sql_llm, text_llm = make_llms(cache=cache)
engine = build_query_engine(sql_llm, synthesize=False)
assistant = FinanceAssistant(sql_llm=sql_llm, text_llm=text_llm, sql_engine=engine)

from IPython.display import display
import plotly.io as pio
pio.renderers.default = "iframe"   # use "iframe" if charts still don't appear on Kaggle

for q in SAMPLE_QUESTIONS:
    r = assistant.ask(q)
    print(f"\nQ: {q}\nSQL: {r.sql}\nrows: {None if r.df is None else len(r.df)} | error: {r.error}")
    print(f"answer: {r.answer[:500]}")
    print(f"llm_calls={r.llm_calls} cache_hits={r.cache_hits} time={r.seconds}s")
    if r.chart_spec:
        print("chart spec:", r.chart_spec)
    if r.fig is not None:
        r.fig.show()
    elif r.df is not None:
        display(r.df)

# Cache check on the chat() path the SQL engine uses
cache.clear()
assistant.reset()
r1 = assistant.ask(SAMPLE_QUESTIONS[0])
assistant.reset()
r2 = assistant.ask(SAMPLE_QUESTIONS[0])
print("run1 calls:", r1.llm_calls, "| run2 calls:", r2.llm_calls, "| run2 hits:", r2.cache_hits)
assert r2.llm_calls == 0 and r2.sql == r1.sql, "cache is not serving repeated questions"


Q: Which 5 companies had the highest net profit margin in 2025?
SQL: SELECT company, ticker, year, net_profit_margin_pct FROM financials WHERE year = 2025 ORDER BY net_profit_margin_pct DESC LIMIT 5
rows: 5 | error: None
answer: The five companies with the highest net profit margin in 2025 were:

1. Nvidia Corp (NVDA) – 55.85 %  
2. Broadcom Inc. (AVGO) – 36.20 %  
3. Microsoft Corp (MSFT) – 36.15 %  
4. Alphabet Inc. (GOOGL) – 32.81 %  
5. Eli Lilly & Co (LLY) – 31.67 %
llm_calls=2 cache_hits=0 time=0.83s


,company,ticker,year,net_profit_margin_pct
0,Nvidia Corp,NVDA,2025,55.85
1,Broadcom Inc.,AVGO,2025,36.20
2,Microsoft Corp,MSFT,2025,36.15
3,Alphabet Inc.,GOOGL,2025,32.81
4,Eli Lilly & Co,LLY,2025,31.67



Q: Plot Microsoft's total revenue and net income from 2021 to 2025
SQL: SELECT year, total_revenue, net_income FROM financials WHERE LOWER(company) LIKE '%microsoft%' AND year BETWEEN 2021 AND 2025 ORDER BY year LIMIT 50
rows: 5 | error: None
answer: Here is your chart (5 data points). The data table and SQL are below.
llm_calls=1 cache_hits=0 time=3.0s
chart spec: {'chart_type': 'line', 'x': 'year', 'y': ['total_revenue', 'net_income'], 'color': None, 'title': "Microsoft's total revenue and net income from 2021 to 2025"}



Q: Which company had the best total stock return in 2025?
SQL: SELECT company, ticker, year, fy_total_return_pct FROM stock_market WHERE year = 2025 ORDER BY fy_total_return_pct DESC LIMIT 1
rows: 1 | error: None
answer: Nvidia Corp (NVDA) had the best total stock return in 2025, with a year‑end return of **133.75 %**.
llm_calls=2 cache_hits=0 time=1.12s


,company,ticker,year,fy_total_return_pct
0,Nvidia Corp,NVDA,2025,133.75



Q: Show a line chart of Nvidia's closing price by year
SQL: SELECT year, close_price_usd FROM stock_market WHERE LOWER(company) LIKE '%nvidia%' ORDER BY year LIMIT 50
rows: 5 | error: None
answer: Here is your chart (5 data points). The data table and SQL are below.
llm_calls=1 cache_hits=0 time=0.96s
chart spec: {'chart_type': 'line', 'x': 'year', 'y': ['close_price_usd'], 'color': None, 'title': "A line chart of Nvidia's closing price by year"}


run1 calls: 2 | run2 calls: 0 | run2 hits: 2


`run2 calls = 0 and run2 hits = 2 shows that cache is working on identical questions.✅`